In [30]:
# crawl_test => 독립적인 모듈 
# VScode , cursor , pycharm , eclipse 등 어떤 것을 쓰게 될 지 모름 
# import crawl_test 라는 식으로 이 코드 파일을 불러와서 사용해야 함(셀 구조가 아닐 때) 
# 실행 여부와 상관없이 이 모듈이 실행됨 

import requests
import re,time
from bs4 import BeautifulSoup as bs
from urllib.parse import urljoin

# 리팩토링 => 한개의 긴 문장 코드를 여러개의 코드로 나누어서 작성.관리하는 프로그래밍 기법 
# 한개의 긴 문장 코드 => 너무 길다보니 버그 발생 => 디버깅 어려움 
# 코드 한개의 실행을 위해 사용해야하는 리소스 많음 => 코드를 분산 = 메모리 스레드 => 멀티스레드 / 싱글스레드 => 운영체제 


BASE = "https://books.toscrape.com/"

def parse_book(card, category) :
    title = card.h3.a["title"].strip()
    product_url = urljoin(BASE,card.h3.a["href"])
    
    price_text = card.select_one(".price_color").get_text(strip=True)
    price_num = re.sub(r'[^0-9.]',"" , price_text)
    price = float(price_num) if price_num else None 
    
    stock_text = card.select_one(".availability").get_text(strip=True) 
    stock = "In stock" in stock_text 
    
    rating_word = card.select_one(".star-rating")["class"][1]
    rating_map = {"One" : 1,"Two" : 2,"Three" : 3,"Four" : 4 ,"Five" : 5}
    rating = rating_map.get(rating_word) 

    return {
        "title" : title,
        "price" : price, 
        "stock" : stock,
        "rating" : rating,
        "category" : category,
        "product_url" : product_url
    }
    
def crawl_category(cat_url,max_pages = 2):
    data = []
    next_url = cat_url
    
    for i in range(max_pages) : 
        html = requests.get(next_url, timeout=10).text
        soup = bs(html,"html.parser")
        category = soup.select_one(".page-header.action h1")
        category = category.get_text(strip=True) if category else "Unknown" # 3항 조건 연산자 
        for card in soup.select("ol.row >li.col-xs-6.col-sm-4.col-md-3.col-lg-3"):
            data.append(parse_book(card,category))
        nxt = soup.select_one("li.next a")
        if not nxt :
            break
        next_url = urljoin(next_url, nxt["href"])
        time.sleep(0.5)
            
    return data 
         
def crawl_few_categories() : 
    html = requests.get(BASE, timeout=10).text
    soup = bs(html,"html.parser")
    cat_links = [urljoin(BASE,a["href"]) for a in soup.select(".side_categories > ul > li > ul > li >a")[:3]]

    all_rows = [] 
    for link in cat_links : 
        all_rows.extend(crawl_category(link)) 
    return all_rows

if __name__ == "__main__" :
    rows = crawl_few_categories() 
    print(f"Collected :{len(rows)}rows") 
    print(rows[0])

Collected :69rows
{'title': "It's Only the Himalayas", 'price': 45.17, 'stock': True, 'rating': 2, 'category': 'Travel', 'product_url': 'https://books.toscrape.com/its-only-the-himalayas_981/index.html'}


In [31]:
from pymongo import MongoClient 

def save_mongo(rows, db_name="playground",col_name="books") : 
    client = MongoClient("mongodb://localhost:27017")
    col = client[db_name][col_name]

    for doc in rows : 
        col.update_one(
            {"title":doc["title"], "product_url":doc["product_url"]}
        )
    print(f"[Mongodb :{len(rows)} rows")